JobHop is a career trajectory dataset, so the cleanest formulation is:
- CV = a person’s past experiences (sequence of ESCO occupations)
- JD = the next occupation they moved into (represented using your ESCO JD text from jd_df)

This becomes a ranking problem:
- Given a CV history, rank candidate job descriptions; the true next job should rank highly.

<a class="anchor" id="chapter1"></a>

# 1. Imports

</a>

In [2]:
from datasets import load_dataset

/opt/anaconda3/envs/ResumeMatcherThesis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [227]:
import pandas as pd
import numpy as np
import os

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [4]:
ds = load_dataset("aida-ugent/JobHop")

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 1341832
    })
    validation: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 167945
    })
    test: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 167924
    })
})

In [ ]:
train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

In [170]:
train_df = train_ds.to_pandas()
val_df = val_ds.to_pandas()
test_df = test_ds.to_pandas()

In [171]:
jd_df = pd.read_csv("../Data/esco_job_descriptions_en.csv")

<a class="anchor" id="chapter3"></a>

# 3. Initial Analysis

</a>

<a class="anchor" id="sub-section-3_1"></a>

## 3.1. Types & Structure

</a>

In [172]:
print("Shape of train df:", train_df.shape)
train_df.head()

Shape of train df: (1341832, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,0,resource manager,Resource managers manage resources for all pot...,1324.8.3,Q1 2016,Q2 2019,True
1,0,health and safety officer,Health and safety officers execute plans for t...,2263.3,Q1 2017,Q2 2019,True
2,0,integration engineer,Integration engineers develop and implement so...,2511.17,Q1 2013,Q1 2016,True
3,0,programme manager,Programme managers coordinate and oversee seve...,1213.4,Q2 2012,Q1 2013,True
4,0,product development engineering drafter,Product development engineering drafters desig...,3118.3.12,Q1 2011,Q2 2012,True


In [173]:
print("Shape of validation df:", val_df.shape)
val_df.head()

Shape of validation df: (167945, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,5,medical administrative assistant,Medical administrative assistants work very cl...,3344.1,Q1 2013,Q4 2013,True
1,5,administrative assistant,Administrative assistants provide administrati...,3343.1,Q4 2013,Q4 2018,True
2,5,room attendant,"Room attendants clean, tidy and restock guest ...",9112.4,None,None,True
3,6,unknown,unknown,None,Q1 2010,Q4 2011,True
4,6,unknown,unknown,None,Q1 2009,None,True


In [174]:
print("Shape of test df:", test_df.shape)
test_df.head()

Shape of test df: (167924, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,16,psychology lecturer,"Psychology lecturers are subject professors, t...",2310.1.35,Q1 2012,Q2 2012,True
1,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q1 2009,Q3 2011,True
2,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q4 2003,Q4 2008,True
3,16,human resources manager,"Human resources managers plan, design and impl...",1212.2,Q3 2000,Q4 2003,True
4,36,leaflet distributor,"Leaflet distributors hand out flyers, leaflet ...",9510.1,Q1 2005,Q4 2006,False


In [175]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341832 entries, 0 to 1341831
Data columns (total 7 columns):
 #   Column               Non-Null Count    Dtype 
---  ------               --------------    ----- 
 0   person_id            1341832 non-null  int64 
 1   matched_label        1341832 non-null  object
 2   matched_description  1341832 non-null  object
 3   matched_code         1330799 non-null  object
 4   start_date           1187091 non-null  object
 5   end_date             1043416 non-null  object
 6   university_studies   1341832 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 62.7+ MB


In [176]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167945 entries, 0 to 167944
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167945 non-null  int64 
 1   matched_label        167945 non-null  object
 2   matched_description  167945 non-null  object
 3   matched_code         166430 non-null  object
 4   start_date           148683 non-null  object
 5   end_date             130359 non-null  object
 6   university_studies   167945 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


In [177]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167924 entries, 0 to 167923
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167924 non-null  int64 
 1   matched_label        167924 non-null  object
 2   matched_description  167924 non-null  object
 3   matched_code         166596 non-null  object
 4   start_date           149029 non-null  object
 5   end_date             131111 non-null  object
 6   university_studies   167924 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


<a class="anchor" id="sub-section-3_2"></a>

## 3.2. Duplicates

</a>

In [178]:
print(train_df[train_df.index.duplicated() == True])
print(val_df[val_df.index.duplicated() == True])
print(test_df[test_df.index.duplicated() == True])

Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []


In [179]:
print("Train: ", train_df.duplicated().mean())
print("Validation: ", val_df.duplicated().mean())
print("Test: ", test_df.duplicated().mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [180]:
train_duplicates = train_df.duplicated().sum()
val_duplicates = val_df.duplicated().sum()
test_duplicates = test_df.duplicated().sum()
print(f"There are {train_duplicates} duplicated rows in train_df.")
print(f"There are {val_duplicates} duplicated rows in val_df.")
print(f"There are {test_duplicates} duplicated rows in test_df.")

There are 35340 duplicated rows in train_df.
There are 4463 duplicated rows in val_df.
There are 4307 duplicated rows in test_df.


In [181]:
print("Train: ", train_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Validation: ", val_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Test: ", test_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [182]:
train_df = train_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
val_df = val_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
test_df = test_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])

<a class="anchor" id="sub-section-3_3"></a>

## 3.3. Missing Values

</a>

In [183]:
train_df.isna().sum()

person_id                   0
matched_label               0
matched_description         0
matched_code            10223
start_date             134229
end_date               274907
university_studies          0
dtype: int64

In [184]:
val_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1377
start_date             16643
end_date               34551
university_studies         0
dtype: int64

In [185]:
test_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1250
start_date             16422
end_date               33955
university_studies         0
dtype: int64

<a class="anchor" id="sub-section-3_4"></a>

## 3.4. Statistical Analysis

</a>

In [186]:
train_df.describe(include = object).T

,count,unique,top,freq
matched_label,1306492,2955,unknown,210964
matched_description,1306492,2955,unknown,210964
matched_code,1296269,2955,unknown,200741
start_date,1172263,227,Q1 2011,61816
end_date,1031585,216,Q4 2012,59030


In [187]:
val_df.describe(include = object).T

,count,unique,top,freq
matched_label,163482,2606,unknown,26531
matched_description,163482,2606,unknown,26531
matched_code,162105,2606,unknown,25154
start_date,146839,203,Q1 2011,7711
end_date,128931,190,Q4 2012,7381


In [188]:
test_df.describe(include = object).T

,count,unique,top,freq
matched_label,163617,2619,unknown,26432
matched_description,163617,2619,unknown,26432
matched_code,162367,2619,unknown,25182
start_date,147195,203,Q1 2011,7639
end_date,129662,194,Q4 2013,7312


In [189]:
print("Unknown rate for train: ", (train_df['matched_code'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_code'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_code'] == 'unknown').mean())

Unknown rate for train:  0.15364885510205956
Unknown rate for validation:  0.15386403396092535
Unknown rate for test:  0.15390821247180916


In [190]:
print("Unknown rate for train: ", (train_df['matched_label'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_label'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_label'] == 'unknown').mean())

Unknown rate for train:  0.16147362555606923
Unknown rate for validation:  0.16228697960631752
Unknown rate for test:  0.16154800540286157


These rows do not have a valid ESCO occupation mapping. The original resume text could not be aligned to any ESCO occupation, therefore: no occupation, no ESCO skills, no ontology relations. These rows are structurally unusable for ESCO-augmented CV–JD matching.

In [191]:
def remove_unknowns(df):
    return df[
        (df["matched_code"] != "unknown") &
        (df["matched_label"] != "unknown")
    ].copy()

In [192]:
train_df = remove_unknowns(train_df)
val_df = remove_unknowns(val_df)
test_df = remove_unknowns(test_df)

In [193]:
print("Unknown rate for train: ", (train_df['matched_code'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_code'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_code'] == 'unknown').mean())

Unknown rate for train:  0.0
Unknown rate for validation:  0.0
Unknown rate for test:  0.0


In [194]:
print("Unknown rate for train: ", (train_df['matched_label'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_label'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_label'] == 'unknown').mean())

Unknown rate for train:  0.0
Unknown rate for validation:  0.0
Unknown rate for test:  0.0


In [195]:
print("Train shape: ", train_df.shape)
train_df.describe(include = object).T

Train shape:  (1095528, 7)


,count,unique,top,freq
matched_label,1095528,2954,administrative assistant,56538
matched_description,1095528,2954,Administrative assistants provide administrati...,56538
matched_code,1095528,2954,3343.1,56538
start_date,962862,224,Q1 2011,50775
end_date,824954,212,Q4 2012,46531


<a class="anchor" id="chapter4"></a>

# 4. Creating Dataset

</a>

<a class="anchor" id="sub-section-4_1"></a>

## 4.1. Logic Check

</a>

**Check people with less than 2 experiences**

In [196]:
train_df.groupby('person_id')['matched_code'].nunique().describe()

count    276692.000000
mean          3.330273
std           1.910435
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          18.000000
Name: matched_code, dtype: float64

In [197]:
print("People with ≥ 2 occupations in train: ", (train_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in val: ", (val_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in test: ", (test_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())

People with ≥ 2 occupations in train:  0.830381796365634
People with ≥ 2 occupations in val:  0.8303811367642384
People with ≥ 2 occupations in test:  0.8304957363780893


For modeling and evaluation, the sample should be restricted to individuals with at least two distinct occupations, as career transitions are required to construct CV–job pairs. Individuals with a single recorded occupation were retained for descriptive analysis but excluded from supervised matching experiments.

In [198]:
def valid_persons(df):
    return df.groupby('person_id')['matched_code'].nunique().loc[lambda x: x >= 2].index

train_df = train_df[train_df['person_id'].isin(valid_persons(train_df))]
val_df = val_df[val_df['person_id'].isin(valid_persons(val_df))]
test_df = test_df[test_df['person_id'].isin(valid_persons(test_df))]

In [199]:
#train_df.groupby('person_id')['matched_code'].nunique().describe()
#val_df.groupby('person_id')['matched_code'].nunique().describe()
test_df.groupby('person_id')['matched_code'].nunique().describe()

count    28731.000000
mean         3.814869
std          1.759112
min          2.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         16.000000
Name: matched_code, dtype: float64

**Check date consistency**

In [200]:
def quarter_to_timestamp(q):
    if pd.isna(q):
        return np.nan
    try:
        quarter, year = q.split()
        year = int(year)
        q_num = int(quarter[1])
        month = (q_num - 1) * 3 + 1
        return pd.Timestamp(year=year, month=month, day=1)
    except:
        return np.nan

In [201]:
train_df['start_ts'] = train_df['start_date'].apply(quarter_to_timestamp)
train_df['end_ts'] = train_df['end_date'].apply(quarter_to_timestamp)

val_df['start_ts'] = val_df['start_date'].apply(quarter_to_timestamp)
val_df['end_ts'] = val_df['end_date'].apply(quarter_to_timestamp)

test_df['start_ts'] = test_df['start_date'].apply(quarter_to_timestamp)
test_df['end_ts'] = test_df['end_date'].apply(quarter_to_timestamp)

In [202]:
train_df[['start_date', 'end_date', 'start_ts', 'end_ts']].sample(10)

,start_date,end_date,start_ts,end_ts
310876,Q1 2015,None,2015-01-01,NaT
908744,Q1 2007,Q4 2010,2007-01-01,2010-10-01
36431,Q1 2014,Q3 2015,2014-01-01,2015-07-01
412347,Q1 2005,Q4 2009,2005-01-01,2009-10-01
1022668,Q1 2014,Q4 2015,2014-01-01,2015-10-01
1286973,Q1 1999,Q4 2001,1999-01-01,2001-10-01
546608,Q2 2016,Q3 2016,2016-04-01,2016-07-01
220852,Q3 2009,Q3 2011,2009-07-01,2011-07-01
632269,Q1 2011,Q4 2011,2011-01-01,2011-10-01
1304241,Q2 2017,Q1 2018,2017-04-01,2018-01-01


In [203]:
print("Invalid rate train: ", (train_df['start_ts'] > train_df['end_ts']).mean())
print("Invalid rate val: ", (val_df['start_ts'] > val_df['end_ts']).mean())
print("Invalid rate test: ", (test_df['start_ts'] > test_df['end_ts']).mean())

Invalid rate train:  1.6440353912465752e-05
Invalid rate val:  1.5473289234459014e-05
Invalid rate test:  1.5443896186129837e-05


Quarter-level dates are converted to timestamps for temporal ordering. Temporal inconsistencies affect fewer than 0.01% of records across all splits and can be tolerated. Career sequences should be ordered by start date, with end date used as a fallback.

<a class="anchor" id="sub-section-4_2"></a>

## 4.2. Constructing CVs

</a>

In [204]:
def dedup_consecutive(seq):
    out = []
    prev = object()
    for x in seq:
        if x != prev:
            out.append(x)
        prev = x
    return out

In [205]:
def build_timelines(df, remove_consecutive_duplicates=True):
    df = df.copy()

    # Sort chronologically within person
    df = df.sort_values(
        by=["person_id", "start_ts", "end_ts"],
        ascending=[True, True, True],
        na_position="last",
    )

    # Aggregate into aligned sequences
    timelines = (
        df.groupby("person_id")
          .apply(lambda g: pd.Series({
              "occupation_codes": g["matched_code"].tolist(),
              "occupation_labels": g["matched_label"].tolist(),
          }))
          .reset_index()
    )

    # Optional: remove consecutive duplicates
    if remove_consecutive_duplicates:
        timelines["occupation_codes"] = timelines["occupation_codes"].apply(dedup_consecutive)
        timelines["occupation_labels"] = timelines["occupation_labels"].apply(dedup_consecutive)

    return timelines


In [206]:
train_timelines = build_timelines(train_df)
val_timelines = build_timelines(val_df)
test_timelines = build_timelines(test_df)

/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_979/1472544440.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_979/1472544440.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_979/1472544440.py:14: FutureWarning: DataFrameGro

In [207]:
assert (train_timelines["occupation_codes"].apply(len)
        == train_timelines["occupation_labels"].apply(len)).all()

train_timelines["occupation_codes"].apply(len).describe()

count    229760.000000
mean          4.101227
std           2.027820
min           2.000000
25%           3.000000
50%           4.000000
75%           5.000000
max          22.000000
Name: occupation_codes, dtype: float64

In [208]:
train_timelines.sample(5)[['person_id', 'occupation_codes', 'occupation_labels']]

,person_id,occupation_codes,occupation_labels
60293,116647,"[5321.1, 5223.4, 5321.1, 3412.4.13, 2341.1]","[healthcare assistant, sales assistant, health..."
127271,246595,"[9111.1, 9412.2, 5131.2, 5223.6]","[domestic cleaner, kitchen porter, waiter/wait..."
150558,291347,"[4211.2, 1420.3]","[post office counter clerk, sales account mana..."
146192,282905,"[4214.1, 4311.1, 9411.2, 3343.1, 5223.6]","[debt collector, billing clerk, quick service ..."
95457,184787,"[5230.1, 7512.1, 5321.1]","[cashier, baker, healthcare assistant]"


<a class="anchor" id="sub-section-4_3"></a>

## 4.3. Constructing CV-Target Occupation pairs

</a>

In [209]:
def history_to_cv_text(labels, max_roles=None):
    """
    labels: list of occupation labels (history)
    max_roles: keep only last k roles to control length (None = keep all)
    """
    if max_roles is not None:
        labels = labels[-max_roles:]
    return "Work experience: " + "; ".join(labels)

In [210]:
def make_positive_pairs(timelines_df, max_history_roles=None):
    """
    timelines_df columns:
      - person_id
      - occupation_codes (list)
      - occupation_labels (list)
    """
    rows = []

    for _, r in timelines_df.iterrows():
        pid = r["person_id"]
        codes = r["occupation_codes"]
        labels = r["occupation_labels"]

        L = len(codes)
        if L < 2:
            continue  # should not happen after filtering

        for t in range(1, L):
            cv_codes = codes[:t]
            cv_labels = labels[:t]

            rows.append({
                "person_id": pid,
                "t": t,
                "cv_codes": cv_codes,
                "cv_labels": cv_labels,
                "cv_text": history_to_cv_text(cv_labels, max_roles=max_history_roles),
                "target_code": codes[t],
                "target_label": labels[t],
            })

    return pd.DataFrame(rows)

In [211]:
train_pairs_pos = make_positive_pairs(train_timelines, max_history_roles=5)
val_pairs_pos = make_positive_pairs(val_timelines, max_history_roles=5)
test_pairs_pos = make_positive_pairs(test_timelines, max_history_roles=5)

In [212]:
len(train_pairs_pos), len(val_pairs_pos), len(test_pairs_pos)

(712538, 89135, 89214)

In [213]:
train_pairs_pos.sample(5)[["cv_text", "target_label"]]

,cv_text,target_label
1347,Work experience: mayor,social entrepreneur
655700,Work experience: social services manager,screen making technician
173850,Work experience: warehouse worker,smart home installer
708053,Work experience: bartender,mobility services manager
9555,Work experience: shelf filler; food service w...,customer service representative


In [214]:
train_pairs_pos.sample(5)

,person_id,t,cv_codes,cv_labels,cv_text,target_code,target_label
491370,306488,1,[9333.2],[distribution centre dispatcher],Work experience: distribution centre dispatcher,1324.3.4,warehouse manager
666638,416032,2,"[5414.1, 9112.2]","[security guard, building cleaner]",Work experience: security guard; building cleaner,5120.1,cook
74604,46482,1,[4110.1],[office clerk],Work experience: office clerk,3343.4,management assistant
257462,160150,1,[3423.2],[fitness instructor],Work experience: fitness instructor,5312.2,primary school teaching assistant
643497,401948,4,"[9329.1, 9321.2, 5246.1, 8322.7]","[factory hand, hand packer, food service work...",Work experience: factory hand; hand packer; f...,5223.4,sales assistant


In [215]:
print("Empty CV in train: ", (train_pairs_pos["cv_text"].str.len() == 0).mean())
print("Empty CV in val: ", (val_pairs_pos["cv_text"].str.len() == 0).mean())
print("Empty CV in test: ", (test_pairs_pos["cv_text"].str.len() == 0).mean())

Empty CV in train:  0.0
Empty CV in val:  0.0
Empty CV in test:  0.0


<a class="anchor" id="sub-section-4_4"></a>

## 4.4. Merge CV-Target Occupation with JDs

</a>

In [216]:
train_pairs_pos[["target_code", "target_label"]].head()

,target_code,target_label
0,3512.1,ICT help desk agent
1,3512.2,ICT help desk manager
2,4222.1,customer contact centre information clerk
3,1324.4,move manager
4,3118.3.12,product development engineering drafter


In [217]:
jd_df[["occupationUri", "iscoGroup", "preferredLabel"]].head()

,occupationUri,iscoGroup,preferredLabel
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager


In [218]:
# Checking if there are no null values (normally done in other notebook, but just in case)
jd_df["essential_skills"].isna().sum()

np.int64(0)

JobHop provides occupation mappings aligned with an earlier ESCO v1.x release. In this work, we use a recent ESCO release to construct job descriptions and ontology features. As ESCO occupation identifiers are largely stable across versions, this does not materially affect the matching task. Minor discrepancies may arise from ESCO version differences between JobHop and the ontology release used, though occupation-level identifiers remain stable.

In [219]:
# Normalize
train_pairs_pos["target_label_key"] = train_pairs_pos["target_label"].str.lower().str.strip()
val_pairs_pos["target_label_key"] = val_pairs_pos["target_label"].str.lower().str.strip()
test_pairs_pos["target_label_key"] = test_pairs_pos["target_label"].str.lower().str.strip()

jd_df["preferredLabel_key"] = jd_df["preferredLabel"].str.lower().str.strip()

In [220]:
jd_df_unique = jd_df.drop_duplicates(subset=["preferredLabel_key"], keep="first")

In [221]:
def merge_with_jd(pairs_pos, jd_df_unique):
    pairs_pos = pairs_pos.copy()
    pairs_pos["target_label_key"] = pairs_pos["target_label"].str.lower().str.strip()

    merged = pairs_pos.merge(
        jd_df_unique[["preferredLabel_key", "jd_text", "occupationUri", "iscoGroup"]],
        left_on="target_label_key",
        right_on="preferredLabel_key",
        how="inner"
    )

    # ensure 1 row per transition
    merged = merged.drop_duplicates(subset=["person_id", "t", "target_label"])

    return merged

Due to label-level alignment between JobHop and ESCO, some occupations mapped to multiple ESCO definitions; in such cases, a single representative job description was retained per transition.

In [222]:
train_pairs = merge_with_jd(train_pairs_pos, jd_df_unique)
val_pairs = merge_with_jd(val_pairs_pos, jd_df_unique)
test_pairs = merge_with_jd(test_pairs_pos, jd_df_unique)

In [223]:
print("Train: ", len(train_pairs) / len(train_pairs_pos))
print("Val: ", len(val_pairs) / len(val_pairs_pos))
print("Test: ", len(test_pairs) / len(test_pairs_pos))

Train:  0.9931161566119983
Val:  0.9927525663319684
Test:  0.9935996592463067


A small fraction of transitions (<1%) could not be matched to ESCO job descriptions and were excluded.

In [224]:
train_pairs.sample(3)[["cv_text", "target_label", "jd_text"]]

,cv_text,target_label,jd_text
92762,Work experience: warehouse order picker; wareh...,sales processor,sales processor. Sales processors handle sales...
503672,Work experience: commercial sales representati...,cook,cook. Cooks are culinary operatives who are ab...
492593,Work experience: telephone switchboard operator,project manager,project manager. Project managers oversee the ...


In [225]:
train_pairs.head()

,person_id,t,cv_codes,cv_labels,cv_text,target_code,target_label,target_label_key,preferredLabel_key,jd_text,occupationUri,iscoGroup
0,0,1,[2353.1],[language school teacher],Work experience: language school teacher,3512.1,ICT help desk agent,ict help desk agent,ict help desk agent,ICT help desk agent. ICT help desk agents prov...,http://data.europa.eu/esco/occupation/aaeec9a7...,3512
1,0,2,"[2353.1, 3512.1]","[language school teacher, ICT help desk agent]",Work experience: language school teacher; ICT ...,3512.2,ICT help desk manager,ict help desk manager,ict help desk manager,ICT help desk manager. ICT help desk managers ...,http://data.europa.eu/esco/occupation/1242d99a...,3512
2,0,3,"[2353.1, 3512.1, 3512.2]","[language school teacher, ICT help desk agent,...",Work experience: language school teacher; ICT ...,4222.1,customer contact centre information clerk,customer contact centre information clerk,customer contact centre information clerk,customer contact centre information clerk. Cus...,http://data.europa.eu/esco/occupation/b7b75eb6...,4222
3,0,4,"[2353.1, 3512.1, 3512.2, 4222.1]","[language school teacher, ICT help desk agent,...",Work experience: language school teacher; ICT ...,1324.4,move manager,move manager,move manager,move manager. Move managers coordinate all the...,http://data.europa.eu/esco/occupation/8f928d49...,1324
4,0,5,"[2353.1, 3512.1, 3512.2, 4222.1, 1324.4]","[language school teacher, ICT help desk agent,...",Work experience: language school teacher; ICT ...,3118.3.12,product development engineering drafter,product development engineering drafter,product development engineering drafter,product development engineering drafter. Produ...,http://data.europa.eu/esco/occupation/c52129d9...,3118


<a class="anchor" id="chapter5"></a>

# 5. Saving Dataset

</a>

In [ ]:
#os.makedirs("/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Data/Processed", exist_ok=True)

In [232]:
#os.getcwd()

'/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Data'

In [ ]:
#os.chdir('/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Data/Processed')

In [ ]:
#train_pairs.to_csv("train_pairs.csv", index=False)
#val_pairs.to_csv("val_pairs.csv", index=False)
#test_pairs.to_csv("test_pairs.csv", index=False)